# Generate Atomic Data Reference for TARDIS
This noteboook generates atomic data to be compared with the [atomic data stored in the tardis regression data repository](https://github.com/tardis-sn/tardis-regression-data/blob/main/atom_data/kurucz_cd23_chianti_H_He.h5) for testing purposes. 

** As of May 14, 2025, the Chianti database exported by Carsus is NOT compatible with versions of TARDIS before v2025.5.18. Use a version of Carsus before this data to generate computed Chianti data suitable for older versions of TARDIS. **


## `kurucz_cd23_chianti_H_He_latest.h5` reproduction status

The TARDIS regression file `kurucz_cd23_chianti_H_He_latest.h5` uses the legacy TARDIS atom-data HDF schema (`database_version='v0.9'`), Kurucz GFALL levels with label-aware level identity, CHIANTI H-He collisions, Knox-Long zeta data, NNDC decay radiation data, and Astropy constants pinned to the CODATA 2010 / IAU 2012 set before importing `astropy.units` or `astropy.constants`.

Current investigation found that the public/local Kurucz GFALL files available on this machine do not exactly reproduce one O IV level near 31.635 eV in the reference output. The checked GFALL files include the Kurucz 2012 download (`md5=2704fbda0b8cba61bb70426234224464`) and `/home/afullard/urilight/AtomicDataBase/gfall.dat` (`md5=8b9d9897a7e8830cfba889084d4653a2`); both contain the same O IV rows in that region. The original NNDC CSV snapshot used for `/decay_radiation_data` was also not found. The local `/home/afullard/Downloads/tardisnuclear/decay_radiation.h5` file contains `/decay_radiation` with shape `(95, 9)`, not the reference `/decay_radiation_data` table with shape `(242295, 41)`. With the available sources, Carsus can reproduce the reference schema and all tables except for that remaining source-data-level mismatch and the NNDC decay table content. Exact binary/table reproduction therefore requires recovering the original GFALL/NNDC source files or documenting an explicit compatibility patch for those historical inputs.


In [ ]:
# Settings used for the legacy TARDIS H-He reference atom-data layout
from carsus.io.kurucz import GFALLReader
from carsus.io.chianti_ import ChiantiReader
gfall_reader = GFALLReader(
    'H-Zn',
    unique_level_identifier=['energy', 'j', 'label'],
)
chianti_reader = ChiantiReader('H-He', collisions=True, priority=20)

# Exact decay reproduction requires the historical carsus-data-nndc/csv snapshot.
# nndc_reader = NNDCReader(dirname='path/to/historical/carsus-data-nndc/csv')

# After constructing TARDISAtomData, write the legacy-compatible schema with:
# atom_data.to_hdf(
#     'kurucz_cd23_chianti_H_He_latest.h5',
#     legacy_tardis_schema=True,
#     database_version='v0.9',
# )


In [ ]:
import pathlib
from carsus.io.nist import NISTWeightsComp, NISTIonizationEnergies

In [ ]:
atomic_weights = NISTWeightsComp()
ionization_energies = NISTIonizationEnergies('H-Zn', )

In [ ]:
from carsus.io.kurucz import GFALLReader

gfall_reader = GFALLReader('H-Zn')

In [ ]:
cmfgen_path = '../../carsus-data-cmfgen/atomic/'
if not pathlib.Path(cmfgen_path).exists():
    cmfgen_path = "/tmp/atomic/"

In [ ]:
from carsus.io.cmfgen import CMFGENReader

cmfgen_reader = CMFGENReader.from_config('Si 0-1',
                                         cmfgen_path,
                                         priority=30,
                                         ionization_energies=True,
                                         cross_sections=True,
                                         collisions=True,
                                         temperature_grid=None,
                                         drop_mismatched_labels=True)


In [ ]:
from carsus.io.zeta import KnoxLongZeta

zeta_data = KnoxLongZeta()

In [ ]:
from carsus.io.output import TARDISAtomData

atom_data = TARDISAtomData(atomic_weights,
                           ionization_energies,
                           gfall_reader,
                           zeta_data,
                           cmfgen_reader=cmfgen_reader)

In [ ]:
atom_data.to_hdf('kurucz_cd23_cmfgen_H_Si.h5')